### 문제
- GridSearchCV와 연동하기 위해서 Word2Vec class 생성한 것과 같이

- 해당 class 보강

- 모델을 선택할수 있도록 생성자 함수 추가적인 작업

    - 3개의 매개변수를 추가
        - min_n(기본값 2), max_n(기본값 4), bucket(기본값 2e+6)
    - 마지막 매개변수 1개 추가
        - model을 선택할 수 있는 매개변수
        - type의 기본값은 'w2v'
- fit함수 수정

    - self.type에 따라서 학습이 되는 모델을 변경
        - 'w2v' 라면 -> Word2Vec 학습하고 self.model에 대입
        - 'ft' 라면 -> FastText 학습하고 self.model에 대입
- 해당 클래스를 모듈화

    - 모듈의 이름은 'gensim_test'
1. 모듈 로드
2. tokenizer는 Okt 사용
3. 데이터 셋은 ratings_test.txt 파일을 로드
4. 결측치 제거
5. 글자 간의 좌우 공백을 제거
6. 빈 테스트 데이터가 document에 존재하는가? 제외
7. 중복되는 document를 제외
8. 상위 데이터 100개를 이용하여 gridsearch를 이용해서 파라미터 조합
    - 파라미터 조합 (벡터화 : min_count은 1로 고정)
        - type : ['w2v', 'ft']
        - vector_size : [80, 100]
    - 파라미터 조합 (학습 모델 : SVC)
        - C : [0.8, 1.0]
    - 계층화 폴드는 5회
9. 하위 데이터 100개를 이용하여 검증 : 분류 레포트를 이용

In [1]:
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import classification_report
from sklearn.svm import SVC
from konlpy.tag import Okt
import pandas as pd 
# 커스텀 모듈에서 class만 로드
from gensim_test import Vectorizer

In [2]:
df = pd.read_csv("../data/ratings_test.txt", sep='\t')
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   id        50000 non-null  int64
 1   document  49997 non-null  str  
 2   label     50000 non-null  int64
dtypes: int64(2), str(1)
memory usage: 1.1 MB


In [3]:
# 결측치 제외 
df.dropna(inplace=True)
# document에서 좌우의 공백을 제거 
df['document'] = df['document'].str.strip()
df.loc[ df['document'] == '',  ]

,id,document,label


In [4]:
# 빈 텍스트 제외
df = df.loc[~(df['document'] == ''), ]

In [5]:
# document의 중복 데이터를 제거
df.drop_duplicates('document', inplace = True)

In [6]:
df.info()

<class 'pandas.DataFrame'>
Index: 49157 entries, 0 to 49999
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   id        49157 non-null  int64
 1   document  49157 non-null  str  
 2   label     49157 non-null  int64
dtypes: int64(2), str(1)
memory usage: 1.5 MB


In [7]:
# 토큰화 함수 생성 
okt = Okt()

tokenizer = lambda x : [ word for word in okt.morphs(x) ]

In [8]:
pipe = Pipeline(
    [
        ('vector', Vectorizer(tokenizer=tokenizer, min_count=1)), 
        ('svc', SVC(random_state=42))
    ]
)

In [9]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [10]:
params = {
    'vector__type' : ['w2v', 'ft'], 
    'vector__vector_size' : [80, 100], 
    'vector__l2' : [False, True],
    'svc__C' : [0.8, 1.0]
}

In [11]:
grid = GridSearchCV(
    estimator= pipe, 
    param_grid= params, 
    cv = cv, 
    verbose=1
)

In [12]:
X_train = df.head(100)['document'].values
y_train = df.head(100)['label'].values
X_test = df.tail(100)['document'].values
y_test = df.tail(100)['label'].values

In [13]:
grid.fit(X_train, y_train)

Fitting 5 folds for each of 16 candidates, totalling 80 fits


c:\study\multicampus_practice\venv\Lib\site-packages\sklearn\model_selection\_validation.py:490: FitFailedWarning: 
40 fits failed out of a total of 80.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
40 fits failed with the following error:
Traceback (most recent call last):
  File "c:\study\multicampus_practice\venv\Lib\site-packages\sklearn\model_selection\_validation.py", line 833, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "c:\study\multicampus_practice\venv\Lib\site-packages\sklearn\base.py", line 1336, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\study\multicampus_practice\venv\Lib\site-packages\sklearn\pipeline.py", line 6

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...m_state=42))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'svc__C': [0.8, 1.0], 'vector__l2': [False, True], 'vector__type': ['w2v', 'ft'], 'vector__vector_size': [80, 100]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",None
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",None
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",StratifiedKFo... shuffle=True)
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time for

In [14]:
print("최적의 모델의 성능 점수 : ", grid.best_score_)

최적의 모델의 성능 점수 :  0.55


In [15]:
pred = grid.predict(X_test)

In [16]:
print(classification_report(pred, y_test))

              precision    recall  f1-score   support

           0       0.78      0.60      0.68        67
           1       0.45      0.67      0.54        33

    accuracy                           0.62       100
   macro avg       0.62      0.63      0.61       100
weighted avg       0.67      0.62      0.63       100

